# Blinkit Price LoRA — Regression Fine-Tune

Fine-tunes `distilbert-base-uncased` with PEFT LoRA to predict Indian e-commerce prices from natural language: `"Product Name, Festival, Platform" → price_inr`.

This is **Arm 2** in the 5-arm price comparison system. Training runs in ~15-20 minutes on a T4 GPU.

**Steps:**
1. Install deps → Mount Drive → W&B login
2. Generate training data (27 products × festivals × platforms)
3. Train LoRA regression head
4. Log metrics to W&B
5. Save adapter → download or push to HF Hub

> **Runtime**: Runtime → Change runtime type → **T4 GPU**

In [1]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers peft accelerate wandb torch

In [2]:
# ── 2. Mount Google Drive (adapter will be saved here) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
ADAPTER_SAVE_PATH = "/content/drive/MyDrive/blinkit_price_lora"
os.makedirs(ADAPTER_SAVE_PATH, exist_ok=True)
print(f"Adapter will be saved to: {ADAPTER_SAVE_PATH}")

Mounted at /content/drive
Adapter will be saved to: /content/drive/MyDrive/blinkit_price_lora


In [ ]:
# ── 3. Weights & Biases login ─────────────────────────────────────────────────
# Get your key from: wandb.ai → User Settings → API Keys
import wandb
wandb.login()  # will prompt for your API key

WANDB_PROJECT = "blinkit-price-lora"

In [ ]:
# ── 4. Configuration ──────────────────────────────────────────────────────────
BASE_MODEL    = "distilbert-base-uncased"
MAX_LENGTH    = 64
EPOCHS        = 15
BATCH_SIZE    = 32
LEARNING_RATE = 2e-3
SEED          = 42
PRICE_SCALE   = 10_000.0   # divide targets by this; multiply predictions back
LORA_R        = 8
LORA_ALPHA    = 16

import random, json, time
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ── 5. Generate training data ─────────────────────────────────────────────────
#
# Format: {"text": "Product, Festival, Platform", "price": <float>}
# Each product gets one row per festival × platform combo with realistic price noise.

# (name, base_price_inr)
PRODUCTS = [
    # Chocolates (real Blinkit catalog BL-*)
    ('Cadbury Dairy Milk Milkinis Milk Chocolate Bar',         36),
    ('Nestle KitKat 4 Fingers Choco Coated Wafer Bar',         32),
    ('Cadbury Dairy Milk Chocolate Bar Cricket Pack',           27),
    ('Cadbury Dairy Milk Silk Oreo Milk Chocolate Bar',        105),
    ('Snickers Peanut Nougat & Caramel Chocolate Filled Bar',  47),
    ('Cadbury Bournville Rich Cocoa 70% Dark Chocolate Bar',   150),
    ('Cadbury Dairy Milk Silk Milk Chocolate Bar',             102),
    ('Kinder Joy Tweety 20 g',                                  52),
    ('Snickers Butterscotch Flavoured Chocolate Filled Bar',    70),
    ('Amul Dark Chocolate Bar',                                 47),
    # Groceries (synthetic SF-* catalog)
    ('Tata Salt Iodised',                                       28),
    ('Aashirvaad Atta Whole Wheat',                            285),
    ('Fortune Sunflower Oil',                                  165),
    ('Saffola Gold Oil',                                       185),
    ('Amul Gold Milk',                                          35),
    ('Maggi 2-Minute Noodles',                                  96),
    ('Tata Tea Premium',                                       290),
    ('Red Label Tea',                                          270),
    ('Surf Excel Matic Liquid',                                530),
    ('Colgate MaxFresh Toothpaste',                             99),
    ('Dettol Handwash Refill',                                 299),
    ('Lays Classic Salted',                                     50),
    # Electronics (synthetic SF-* catalog)
    ('Apple iPhone 15 128GB',                                79900),
    ('Samsung Galaxy S24 256GB',                             89999),
    ('OnePlus Nord CE4 256GB',                               26999),
    ('Redmi Note 13 Pro 128GB',                              25999),
    ('boAt Airdopes 161 TWS',                                 2990),
    ('Sony WH-CH520 Headphones',                              4990),
    ('Mi Smart Band 8',                                       3499),
    ('HP 15s Ryzen 5 Laptop',                                58999),
    ('Logitech M331 Silent Mouse',                              995),
    ('TP-Link Archer C6 Router',                              2999),
    # Personal care & baby
    ('Nivea Body Lotion',                                      425),
    ('Head & Shoulders Shampoo',                               540),
    ('Pampers Diapers Medium',                                 799),
]

# (festival_name, discount_bias)
FESTIVALS = [
    ("No Festival",         0.00),
    ("Republic Day Sale",   0.18),
    ("Holi",                0.10),
    ("Summer Sale",         0.12),
    ("Independence Day",    0.20),
    ("Onam",                0.14),
    ("Big Billion Days",    0.30),
    ("Great Indian Festival",0.30),
    ("Dhanteras",           0.22),
    ("Diwali",              0.28),
    ("Black Friday",        0.25),
    ("Year End Sale",       0.16),
]

PLATFORMS = ["Blinkit", "Zepto", "Amazon.in", "Flipkart"]

rng = random.Random(SEED)
rows = []
for name, base_price in PRODUCTS:
    for fest_name, bias in FESTIVALS:
        for platform in PLATFORMS:
            noise = rng.uniform(-0.05, 0.05)           # ±5% noise
            price = base_price * (1.0 - bias) * (1.0 + noise)
            price = max(1.0, round(price, 2))
            rows.append({"text": f"{name}, {fest_name}, {platform}", "price": price})

rng.shuffle(rows)
split = int(len(rows) * 0.8)
train_rows, test_rows = rows[:split], rows[split:]

with open("/content/price_train.jsonl", "w") as f:
    f.write("\n".join(json.dumps(r) for r in train_rows))
with open("/content/price_test.jsonl", "w") as f:
    f.write("\n".join(json.dumps(r) for r in test_rows))

print(f"Total rows: {len(rows)} | Train: {len(train_rows)} | Test: {len(test_rows)}")
print("Sample:", train_rows[0])

In [ ]:
# ── 6. Dataset class ──────────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class PriceDataset(Dataset):
    def __init__(self, rows, tokenizer):
        texts  = [r["text"] for r in rows]
        prices = [r["price"] / PRICE_SCALE for r in rows]  # normalize
        enc = tokenizer(texts, truncation=True, padding=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
        # regression: labels shape (N,) float
        enc["labels"] = torch.tensor(prices, dtype=torch.float)
        self.enc = enc

    def __len__(self):
        return len(self.enc["labels"])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.enc.items()}


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
train_ds  = PriceDataset(train_rows, tokenizer)
test_ds   = PriceDataset(test_rows,  tokenizer)
train_dl  = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dl   = DataLoader(test_ds,  batch_size=BATCH_SIZE)
print(f"Train batches: {len(train_dl)} | Test batches: {len(test_dl)}")

In [ ]:
# ── 7. Build LoRA model ───────────────────────────────────────────────────────
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    problem_type="regression",   # MSE loss, float labels
)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_lin", "v_lin"],   # distilbert attention projections
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
)
model = get_peft_model(base, lora_cfg)
model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
# ── 8. Training loop ──────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE
)

run = wandb.init(
    project=WANDB_PROJECT,
    name="lora-price-regression",
    config={
        "base_model":    BASE_MODEL,
        "epochs":        EPOCHS,
        "batch_size":    BATCH_SIZE,
        "lr":            LEARNING_RATE,
        "lora_r":        LORA_R,
        "lora_alpha":    LORA_ALPHA,
        "price_scale":   PRICE_SCALE,
        "n_train":       len(train_rows),
        "n_test":        len(test_rows),
        "trainable_params": trainable,
    }
)


def eval_mae(loader):
    model.eval()
    abs_errors = []
    with torch.no_grad():
        for batch in loader:
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            labels  = batch.pop("labels")
            logits  = model(**batch).logits.squeeze(-1)
            preds   = torch.abs(logits) * PRICE_SCALE   # denormalize
            targets = labels * PRICE_SCALE
            abs_errors.extend((preds - targets).abs().tolist())
    return float(np.mean(abs_errors))


t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss, n = 0.0, 0
    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach())
        n += 1

    train_mae = eval_mae(train_dl)
    test_mae  = eval_mae(test_dl)
    metrics   = {
        "train_loss": round(total_loss / n, 6),
        "train_mae_inr": round(train_mae, 2),
        "test_mae_inr":  round(test_mae, 2),
        "epoch": epoch + 1,
    }
    wandb.log(metrics, step=epoch)
    if (epoch + 1) % 3 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | loss={metrics['train_loss']:.5f} "
              f"| train_MAE=₹{train_mae:.2f} | test_MAE=₹{test_mae:.2f}")

wall = round(time.time() - t0, 1)
print(f"\nDone in {wall}s | Final test MAE: ₹{test_mae:.2f}")

wandb.summary.update({"final_test_mae_inr": test_mae, "wall_clock_s": wall})

In [ ]:
# ── 9. Save adapter + tokenizer ───────────────────────────────────────────────
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

# Store PRICE_SCALE so inference code can invert it
meta = {
    "price_scale":   PRICE_SCALE,
    "base_model":    BASE_MODEL,
    "test_mae_inr":  round(test_mae, 2),
    "n_train":       len(train_rows),
    "n_test":        len(test_rows),
}
with open(f"{ADAPTER_SAVE_PATH}/price_lora_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Adapter saved to: {ADAPTER_SAVE_PATH}")
print(json.dumps(meta, indent=2))
wandb.finish()

In [ ]:
# ── 10. Quick smoke test ──────────────────────────────────────────────────────
def predict_price(product_name, festival="No Festival", platform="Blinkit"):
    model.eval()
    text = f"{product_name}, {festival}, {platform}"
    enc  = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    enc  = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logit = model(**enc).logits[0, 0].item()
    return round(abs(logit) * PRICE_SCALE, 2)

tests = [
    ("boAt Airdopes 161 TWS Earbuds", "Diwali",        "Blinkit"),
    ("boAt Airdopes 161 TWS Earbuds", "No Festival",   "Blinkit"),
    ("Cadbury Dairy Milk Milkinis 34g", "Big Billion Days", "Amazon.in"),
    ("Amul Taaza Toned Milk 1L",       "No Festival",   "Zepto"),
    ("Samsung Galaxy Buds FE",          "Diwali",        "Flipkart"),
]

print("Smoke test predictions:")
for product, festival, platform in tests:
    price = predict_price(product, festival, platform)
    print(f"  {product[:40]:40s} | {festival:20s} | ₹{price}")

In [ ]:
# ── 11. (Optional) Push adapter to Hugging Face Hub ──────────────────────────
# Uncomment and set your HF token to make the adapter publicly available.
# After pushing, set HF_PRICE_LORA_REPO in your .env to load it remotely.

# HF_TOKEN = "hf_..."   # from huggingface.co/settings/tokens
# REPO_ID  = "YourUsername/blinkit-price-lora"  # change to your HF username
#
# from huggingface_hub import HfApi
# api = HfApi(token=HF_TOKEN)
# api.create_repo(REPO_ID, exist_ok=True)
# api.upload_folder(folder_path=ADAPTER_SAVE_PATH, repo_id=REPO_ID)
# print(f"Pushed to: https://huggingface.co/{REPO_ID}")

## After Training

1. Download the adapter folder from Drive: `blinkit_price_lora/`
2. Copy it to `capstone/data/price_lora_adapter/` on your local machine
3. The app auto-detects it via `infer_price_lora.adapter_exists()`
4. Run `python -m app.ui` — Arm 2 (LoRA) will now show predictions in the comparison table

**W&B:** Your run is at `wandb.ai → blinkit-price-lora` project. Compare `test_mae_inr` across epochs to see when the model converges.